## Mediator

---

> **In one line.** When $n$ components each talk to every other, the system carries $\binom{n}{2}=\tfrac{n(n-1)}{2}$ direct links — a fully connected graph. A mediator $M$ collapses this into a **star**: every component $C_i$ speaks only to $M$, leaving exactly $n$ links and the rule $C_i \not\rightarrow C_j$, only $C_i \rightarrow M \rightarrow C_j$.

### 1. The components and the hub

Let the system be built from $n$ **components** $C_1, C_2, \dots, C_n$ — the interacting participants, such as $n = 5$ users sharing a chat room or $n = 3$ widgets ($C_1$ a checkbox, $C_2$ a text input, $C_3$ a submit button) inside one form. Each $C_i$ is a single participant that produces events and reacts to them.

Against this set of components sits a distinguished object $M$, the **mediator**: the central hub that every component communicates through. Unlike the components, $M$ holds references to all of the $C_i$, so it alone knows the full roster. The design question is purely topological — *who is allowed to hold a reference to whom* — and the answer is captured by counting edges in the communication graph.

### 2. Why the direct graph explodes

Suppose components communicate **directly**, each $C_i$ permitted to call any $C_j$. The underlying graph is then fully connected, and the number of undirected links is the count of unordered pairs:

$$\text{edges}_{\text{direct}} \;=\; \binom{n}{2} \;=\; \frac{n(n-1)}{2}.$$

This grows quadratically. For $n = 5$ it is already $10$ direct dependencies; for $n = 10$ it is $45$. Worse, the graph is **fragile under growth**: introducing one new component $C_{n+1}$ that must reach everyone adds $n$ new edges at once — one to each existing component. The coupling cost of a single addition scales with the size of the system.

### 3. The star collapse

The mediator removes every component-to-component edge and replaces it with a single spoke from each component to the hub. Formally we forbid all direct links and route every interaction through $M$:

$$\forall\, i \neq j,\quad C_i \not\rightarrow C_j \qquad\text{and}\qquad \forall\, i,\quad C_i \rightarrow M \rightarrow C_j.$$

No $C_i$ holds a reference to any other $C_j$; each holds a reference only to $M$. The communication graph becomes a **star topology** with one spoke per component, so the edge count drops to

$$\text{edges}_{\text{mediated}} \;=\; n.$$

Comparing the two regimes gives the defining reduction of the pattern:

$$\boxed{\,\binom{n}{2} = \frac{n(n-1)}{2} \;=\; O(n^2) \;\;\xrightarrow{\;\text{introduce } M\;}\;\; n = O(n)\,}$$

A quadratic web of mutual dependencies becomes a linear bundle of spokes.

**Direction matters.** A message never crosses directly from sender to recipient. It travels *up* to the hub, where coordination logic lives, and then *down* to whichever recipients $M$ selects:

$$\underbrace{C_i}_{\text{sender}} \;\xrightarrow{\;\text{report event}\;}\; \underbrace{M}_{\text{hub decides}} \;\xrightarrow{\;\text{forward}\;}\; \underbrace{C_j}_{\text{recipient(s)}} \qquad (j \neq i)$$

Contrast the two topologies directly: the direct regime is $C_i \xrightarrow{\;\text{call}\;} C_j$ for every pair, while the mediated regime is $C_i \xrightarrow{\;} M \xrightarrow{\;} C_j$ with the middle stop mandatory.

### 4. Key conditions

1. **Zero direct links.** For all $i \neq j$ we require $C_i \not\rightarrow C_j$: no component holds a reference to, imports, or calls any other component. Each $C_i$ knows only $M$.
2. **$O(n^2) \rightarrow O(n)$ reduction.** Adding a new component $C_{n+1}$ costs exactly **one** new edge — the spoke $C_{n+1} \rightarrow M$ — not $n$ new edges to every existing component. Growth is linear, not quadratic.
3. **Mediator owns coordination.** $M$ decides which components receive which messages. The components are passive in routing: they merely report events to $M$ and react when $M$ calls them. All forwarding policy $C_i \rightarrow M \rightarrow C_j$ lives inside the hub.

&nbsp;

> ✈️ Air traffic control. Without it, every plane must coordinate with every other — $\binom{n}{2}$ channels crowding the sky. With the control tower ($M$), every plane speaks only to the tower: $n$ channels total. Planes never talk to each other directly; the tower decides who hears what.

### Exercise 15 — Chat Room

---

**Scenario:** $n$ users ($C_i$). When Alice sends a message, all others receive it. Users must not hold references to each other — $C_i \not\rightarrow C_j$. All communication flows $C_i \rightarrow M \rightarrow C_j$.

**Your task:** Build `ChatRoom` ($M$) and `User` ($C_i$). Alice calls `self._room.broadcast(self, msg)` — never `bob.receive(msg)`.

```python
room = ChatRoom()                 # M
alice = User("Alice", room)       # C_1 -> M only
bob = User("Bob", room)           # C_2 -> M only
alice.send("Hello!")              # C_1 -> M -> C_2 (not C_1 -> C_2)
```

**Hints**

- $M$ holds `self._users = []`. Each $C_i$ holds `self._room = M`.
- `broadcast(sender, msg)` loops over all users except `sender` — this is $M \rightarrow C_j$ for all $j \neq i$.

In [ ]:
# --------------------------------
# Mediator M — the central hub; holds references to all C_i

class ChatRoom:
    def __init__(self):
        self._users = []                     # M holds all C_i

    def register(self, user):                # add a C_i (one new edge to M)
        self._users.append(user)

    def broadcast(self, sender, msg):        # M -> C_j for all j != i
        # loop over all users except sender, call C_j.receive(...)
        # this is the only place messages are routed: C_i -> M -> C_j
        ...

# --------------------------------
# Component C_i — knows only M, never another C_j

class User:
    def __init__(self, name, room):
        self._name = name
        self._room = room                    # C_i -> M only
        room.register(self)

    def send(self, msg):                     # C_i -> M
        print(f"{self._name} sends: {msg}")
        # route through M, never call another user directly
        ...

    def receive(self, sender, msg):          # M -> C_i
        print(f"  {self._name} received from {sender._name}: {msg}")

# --------------------------------
room = ChatRoom()                 # M
alice = User("Alice", room)       # C_1 -> M only
bob = User("Bob", room)           # C_2 -> M only
carol = User("Carol", room)       # C_3 -> M only
alice.send("Hello!")              # C_1 -> M -> C_2, C_3

### Exercise 16 — UI Form Mediator

---

**Scenario:** Three components: `Checkbox` ($C_1$), `TextInput` ($C_2$), `SubmitButton` ($C_3$). When $C_1$ changes, $M$ disables $C_2$ and $C_3$. Without $M$: $\binom{3}{2} = 3$ direct links. With $M$: 3 links to $M$.

**Your task:** Build `FormMediator` ($M$) coordinating all three. Each $C_i$ notifies $M$ on change; $M$ decides what to update.

```python
mediator = FormMediator()                 # M
checkbox = Checkbox(mediator)             # C_1 -> M
text = TextInput(mediator)                # C_2 -> M
button = SubmitButton(mediator)           # C_3 -> M
checkbox.toggle()                         # C_1 -> M -> C_2, C_3
```

**Hints**

- Each $C_i$ calls `self._mediator.notify(self, event)`. $M$ receives the event and calls the appropriate methods on the other components — $M \rightarrow C_j$.
- Components never import each other ($C_i \not\rightarrow C_j$).

In [ ]:
# --------------------------------
# Mediator M — owns coordination; decides which C_j to update

class FormMediator:
    def __init__(self):
        self.checkbox = None                 # M holds all C_i
        self.text = None
        self.button = None

    def notify(self, sender, event):         # C_i -> M -> C_j
        # if the checkbox (C_1) changed, M disables C_2 and C_3
        # M decides what to update; components never decide for each other
        ...

# --------------------------------
# Components C_i — each knows only M

class Checkbox:
    def __init__(self, mediator):
        self._mediator = mediator            # C_1 -> M
        mediator.checkbox = self
        self.checked = False

    def toggle(self):
        self.checked = not self.checked
        print(f"Checkbox toggled -> {self.checked}")
        self._mediator.notify(self, "toggle")   # C_1 -> M

class TextInput:
    def __init__(self, mediator):
        self._mediator = mediator            # C_2 -> M
        mediator.text = self
        self.enabled = True

    def set_enabled(self, value):            # called by M -> C_2
        self.enabled = value
        print(f"  TextInput enabled -> {self.enabled}")

class SubmitButton:
    def __init__(self, mediator):
        self._mediator = mediator            # C_3 -> M
        mediator.button = self
        self.enabled = True

    def set_enabled(self, value):            # called by M -> C_3
        self.enabled = value
        print(f"  SubmitButton enabled -> {self.enabled}")

# --------------------------------
mediator = FormMediator()                 # M
checkbox = Checkbox(mediator)             # C_1 -> M
text = TextInput(mediator)                # C_2 -> M
button = SubmitButton(mediator)           # C_3 -> M
checkbox.toggle()                         # C_1 -> M -> C_2, C_3